# Imports

In [ ]:
import pandas as pd 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


# Loading data and exploring it 

In [104]:
train_df =pd.read_csv("D:/UNEEQ Interns/Customer chrun/customer_churn_dataset-training-master.csv")
test_df = pd.read_csv("D:/UNEEQ Interns/Customer chrun/customer_churn_dataset-testing-master.csv")

# Handle Churn column properly (fill NaN and convert to int)
train_df['Churn'] = train_df['Churn'].fillna(0).astype(int)
test_df['Churn'] = test_df['Churn'].astype(int)

In [105]:
# Training data
print("First 5 rows of the training data:")
train_df.head()

First 5 rows of the training data:


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1


In [106]:
train_df.describe()

,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn
count,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440833.000000
mean,225398.667955,39.373153,31.256336,15.807494,3.604437,12.965722,631.616223,14.480868,0.567106
std,129531.918550,12.442369,17.255727,8.586242,3.070218,8.258063,240.803001,8.596208,0.495477
min,2.000000,18.000000,1.000000,1.000000,0.000000,0.000000,100.000000,1.000000,0.000000
25%,113621.750000,29.000000,16.000000,9.000000,1.000000,6.000000,480.000000,7.000000,0.000000
50%,226125.500000,39.000000,32.000000,16.000000,3.000000,12.000000,661.000000,14.000000,1.000000
75%,337739.250000,48.000000,46.000000,23.000000,6.000000,19.000000,830.000000,22.000000,1.000000
max,449999.000000,65.000000,60.000000,30.000000,10.000000,30.000000,1000.000000,30.000000,1.000000


In [107]:
train_df['Churn'].value_counts()
train_df.isnull().sum()
train_df.info



<bound method DataFrame.info of         CustomerID   Age  Gender  Tenure  Usage Frequency  Support Calls  \
0              2.0  30.0  Female    39.0             14.0            5.0   
1              3.0  65.0  Female    49.0              1.0           10.0   
2              4.0  55.0  Female    14.0              4.0            6.0   
3              5.0  58.0    Male    38.0             21.0            7.0   
4              6.0  23.0    Male    32.0             20.0            5.0   
...            ...   ...     ...     ...              ...            ...   
440828    449995.0  42.0    Male    54.0             15.0            1.0   
440829    449996.0  25.0  Female     8.0             13.0            1.0   
440830    449997.0  26.0    Male    35.0             27.0            1.0   
440831    449998.0  28.0    Male    55.0             14.0            2.0   
440832    449999.0  31.0    Male    48.0             20.0            1.0   

        Payment Delay Subscription Type Contract Length

In [108]:
train_df.isnull().sum()
train_df.dtypes

CustomerID           float64
Age                  float64
Gender                object
Tenure               float64
Usage Frequency      float64
Support Calls        float64
Payment Delay        float64
Subscription Type     object
Contract Length       object
Total Spend          float64
Last Interaction     float64
Churn                  int64
dtype: object

In [109]:
# Check unique values in 'Churn' before any changes
print("Unique values in 'Churn' before processing:")
print(train_df['Churn'].unique())
print("Value counts:")
print(train_df['Churn'].value_counts())

Unique values in 'Churn' before processing:
[1 0]
Value counts:
Churn
1    249999
0    190834
Name: count, dtype: int64


# Data and feature Engineering 

In [110]:
# Handling missing values for numerical columns by filling them with the mean of each column
train_df.fillna(train_df.median(numeric_only= True), inplace=True)
test_df.fillna(train_df.median(numeric_only= True), inplace=True)

#filling missing values for categorical columns with "Na"
train_df.fillna("Na", inplace=True)
test_df.fillna("Na", inplace=True)


In [111]:
#Encoding categorical variables using label encoder (exclude Churn)

for col in train_df.select_dtypes(include=['object']).columns:
    if col == 'Churn':
        continue  # Skip encoding the target variable
    # Ensure the column is string type to avoid mixed types
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)
    
    # Combine unique values from train and test to fit encoder
    all_values = pd.concat([train_df[col], test_df[col]]).unique()
    
    # Create a new encoder for each column and fit on all values
    le = LabelEncoder()
    le.fit(all_values)
    
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

In [139]:
#Splitting features and target variable (drop CustomerID as it's not predictive)
x = train_df.drop(['Churn', 'CustomerID'], axis=1)
y = train_df['Churn']

# Check class distribution
print("Class distribution in y:")
print(y.value_counts())

# Skip oversampling, use class_weight in the model
x_res, y_res = x, y

Class distribution in y:
Churn
1    249999
0    190834
Name: count, dtype: int64


# Modeling 

In [ ]:
x_train , x_val , y_train , y_val = train_test_split(x_res, y_res, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)

# Train the Random Forest model with balanced class weights
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(x_train_scaled, y_train)

In [124]:
# Predict on the validation set
y_pred = model.predict(x_val_scaled)

# Evaluate the model
print(classification_report(y_val, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     49962
           1       1.00      1.00      1.00     50038

    accuracy                           1.00    100000
   macro avg       1.00      1.00      1.00    100000
weighted avg       1.00      1.00      1.00    100000



In [137]:
# Prepare test data (drop target column and any prediction columns)
columns_to_drop = ['Churn', 'CustomerID']
if 'Predicted_Churn' in test_df.columns:
    columns_to_drop.append('Predicted_Churn')
x_test = test_df.drop(columns_to_drop, axis=1)
x_test_scaled = scaler.transform(x_test)

# Predict on the test set with probability scores
test_probabilities = model.predict_proba(x_test_scaled)
# Use a higher threshold for predicting churn (more conservative)
threshold = 0.95  # Increase this to reduce false positives further
test_predictions = (test_probabilities[:, 1] >= threshold).astype(int)

# Output the feature labels
print("Features used for prediction:")
print(list(x.columns))

# Add predictions as a new column to test_df
test_df['Predicted_Churn'] = test_predictions

# Compare predictions with actual values
print("\nFirst 10 rows comparison:")
print(test_df[['Churn', 'Predicted_Churn']].head(10))

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_df['Churn'], test_df['Predicted_Churn'])
print("\nConfusion Matrix:")
print("[[TN, FP]")
print(" [FN, TP]]")
print(cm)

# Detailed comparison for first 20 rows
print("\nDetailed comparison (first 20 rows):")
comparison_df = test_df[['Churn', 'Predicted_Churn']].copy()
comparison_df['Correct'] = comparison_df['Churn'] == comparison_df['Predicted_Churn']
print(comparison_df.head(20))

# Show first 20 rows of test_df with all columns
print("\nFirst 20 rows of test_df with predictions:")
print(test_df.head(20))

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test_df['Churn'], test_df['Predicted_Churn'])
print(f"\nAccuracy on test set: {accuracy:.4f}")

# Additional metrics
from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(test_df['Churn'], test_df['Predicted_Churn'])
recall = recall_score(test_df['Churn'], test_df['Predicted_Churn'])
f1 = f1_score(test_df['Churn'], test_df['Predicted_Churn'])
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Output the predictions
print("\nTest Predictions:")
print(test_predictions)

# Summary of predictions
import pandas as pd
print("\nPrediction Summary:")
pred_counts = pd.Series(test_predictions).value_counts()
print(pred_counts)

# Show that it's not all 1s
total_predictions = len(test_predictions)
ones_count = pred_counts.get(1, 0)
zeros_count = pred_counts.get(0, 0)
print(f"\nOut of {total_predictions} predictions:")
print(f"- {ones_count} predicted as 1 (churn)")
print(f"- {zeros_count} predicted as 0 (no churn)")
print(f"- Percentage of 1s: {ones_count/total_predictions*100:.1f}%")
print(f"- Percentage of 0s: {zeros_count/total_predictions*100:.1f}%")

# Feature importance
print("\nFeature Importances:")
for feature, importance in zip(x.columns, model.feature_importances_):
    print(f"{feature}: {importance:.4f}")


Features used for prediction:
['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction']

First 10 rows comparison:
   Churn  Predicted_Churn
0      1                1
1      0                1
2      0                1
3      0                1
4      0                1
5      0                1
6      1                1
7      0                1
8      0                1
9      0                1

Confusion Matrix:
[[TN, FP]
 [FN, TP]]
[[ 3661 30220]
 [   90 30403]]

Detailed comparison (first 20 rows):
    Churn  Predicted_Churn  Correct
0       1                1     True
1       0                1    False
2       0                1    False
3       0                1    False
4       0                1    False
5       0                1    False
6       1                1     True
7       0                1    False
8       0                1    False
9       0                1    Fa

In [138]:
# Confusion Matrix for test data
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_df['Churn'], test_df['Predicted_Churn'])
print("Confusion Matrix:")
print(cm)
print("\nTrue Negatives (TN):", cm[0,0])
print("False Positives (FP):", cm[0,1])
print("False Negatives (FN):", cm[1,0])
print("True Positives (TP):", cm[1,1])

Confusion Matrix:
[[ 3661 30220]
 [   90 30403]]

True Negatives (TN): 3661
False Positives (FP): 30220
False Negatives (FN): 90
True Positives (TP): 30403
